# 구글 마운트

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# merge 파일 만들기

In [ ]:
import pandas as pd
import os
import re
import unicodedata
from google.colab import drive

YEAR = 2026

def merge_excel_files(root_dir: str, output_file: str):
    excel_files = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        folder_name = unicodedata.normalize('NFC', os.path.basename(dirpath))
        if re.search(r'\d+월', folder_name):
            for f in filenames:
                if f.endswith('.xlsx') or f.endswith('.xls'):
                    excel_files.append(os.path.join(dirpath, f))

    if not excel_files:
        print("엑셀 파일을 찾을 수 없습니다.")
        return

    all_dfs = []
    for file in sorted(excel_files):
        print(f"읽는 중: {file}")
        try:
            stem = unicodedata.normalize('NFC', os.path.splitext(os.path.basename(file))[0])
            match = re.match(r'(\d{4})_(\d{2})_([a-zA-Z]+)', stem)
            file_name = f"{match.group(1)}_{match.group(2)}_{match.group(3)}" if match else stem
            sheets = pd.read_excel(file, sheet_name=None)
            for sheet_name, df in sheets.items():
                df.insert(0, "file_name", file_name)
                all_dfs.append(df)
        except Exception as e:
            print(f"  ⚠️ 오류 발생: {e}")

    if not all_dfs:
        print("읽을 수 있는 데이터가 없습니다.")
        return

    merged = pd.concat(all_dfs, ignore_index=True)

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    merged.to_excel(output_file, index=False)
    print(f"\n✅ 완료! 총 {len(excel_files)}개 파일 → {output_file}")
    print(f"   행 수: {len(merged):,} / 열 수: {len(merged.columns)}")

import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

merge_excel_files(
    root_dir=f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}",
    output_file=f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수/{YEAR}_merge.xlsx"
)

# 쉼표 삭제 (대수비 행이나, 쉼표 있는 행 삭제)

In [ ]:
import pandas as pd
import re
from google.colab import drive

import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

file_path = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수/{YEAR}_merge.xlsx"
df1 = pd.read_excel(file_path)

remove_always = ['대수비', '대타']
prefixes = ['포구실책', '송구실책', '실책', '포일', '폭투', '보크']
NATOUT_SUFFIXES = ['폭투', '포일', '송구실책', '포구실책', '실책']

def is_prefix_only(p, prefixes):
    base = re.sub(r'\[.*?\]', '', p).strip()
    return any(pf in base for pf in prefixes)

def clean_result(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    value = value.replace('.', ',').replace(';', ',')
    has_comma = ',' in value

    if has_comma:
        parts = [p.strip() for p in value.split(',')]

        # 대수비/대타 제거
        only_remove_always = all(any(r in p for r in remove_always) for p in parts)
        if only_remove_always:
            return parts[0]
        parts_cleaned = [p for p in parts if not any(r in p for r in remove_always)]
        if not parts_cleaned:
            return value
        parts = parts_cleaned

        # 낫아웃 포함 여부 확인
        natout_part = next((p for p in parts if '낫아웃' in p), None)

        if natout_part:
            # 낫아웃+,폭투 / 낫아웃+,포일 형태 → 낫아웃+폭투로 합치기
            if natout_part in ('낫아웃+',):
                suffix_part = next(
                    (p for p in parts if p != natout_part and
                     any(s in p for s in NATOUT_SUFFIXES)), None
                )
                if suffix_part:
                    for s in NATOUT_SUFFIXES:
                        if s in suffix_part:
                            return f'낫아웃+{s}'
                return '낫아웃+'

            # 낫아웃+폭투,도루 / 낫아웃+폭투,주자아웃 등 → 낫아웃+폭투만 남김
            if any(s in natout_part for s in NATOUT_SUFFIXES) or '낫아웃-' in natout_part:
                return natout_part

            # 낫아웃- 단독 또는 나머지
            return natout_part

        # 낫아웃 없는 경우 기존 로직
        non_prefix = [p for p in parts if not is_prefix_only(p, prefixes)]
        return non_prefix[0] if non_prefix else parts[0]

    else:
        if any(r in value for r in remove_always):
            return float('nan')
        return value

# ── 단독 대수비 행 삭제
solo_daesubi_mask = df1['result'].str.strip() == '대수비'
print(f"삭제된 단독 대수비 행 수: {solo_daesubi_mask.sum()}")
df2 = df1[~solo_daesubi_mask].copy().reset_index(drop=True)

# ── result 클리닝
df2['result'] = df2['result'].apply(clean_result)

output_path = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수/cleaned_{YEAR}_merge.xlsx"
df2.to_excel(output_path, index=False)
print(f"완료! {output_path} 저장됨")
print(f"전체 행 수: {len(df1)} → {len(df2)}")
print(f"처리 후 대수비 잔존 행 수: {df2['result'].str.contains('대수비', na=False).sum()}")

# 낫아웃 결과 확인
natout_results = df2[df2['result'].astype(str).str.contains('낫아웃', na=False)]['result'].value_counts()
print("\n=== 낫아웃 result 종류 ===")
print(natout_results.to_string())

# plate_seq 재정의 및 개명 선수와 등번호 변경

In [ ]:
import pandas as pd
import os
import re
from google.colab import drive

import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# ============================================================
# 경로 설정
# ============================================================
DRIVE_PATH  = f'/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수/'
INPUT_FILE  = os.path.join(DRIVE_PATH, f'cleaned_{YEAR}_merge.xlsx')
OUTPUT_FILE = os.path.join(DRIVE_PATH, f'cleaned_{YEAR}_merge.xlsx')

# ============================================================
# 데이터 로드
# ============================================================
df = pd.read_excel(INPUT_FILE)
print(f"로드 완료: {df.shape[0]}행 x {df.shape[1]}열")

# ============================================================
# 선수 개명 매핑 (구이름 → 새이름)
# ============================================================
RENAME_MAP = {
    # ('마산용마고', '손재효(33)'): '손율기(33)',
    # ('경민IT고', '정택준(2)'): '정택준(7)',
    # ('전주고', '이현우(25)'): '이서준(25)',
    # ('서울자동차고', '유석현(22)'): '유석현(30)',
    # ('서울자동차고', '조민성(24)'): '조민성(32)',
    # ('인창고', '김보선(33)'): '김보선(12)',
    # ('밀양BC', '김윤우(7)'): '강윤우(7)',
    # ('예일메디텍고', '이상민(24)'): '이도경(24)',
    # 추가 개명 선수 발견 시 여기에 추가
}

def apply_rename(df, rename_map):
    for (team, old_name), new_name in rename_map.items():
        mask = (df['team'] == team) & (df['batter'] == old_name)
        renamed = mask.sum()
        if renamed > 0:
            df.loc[mask, 'batter'] = new_name
            print(f"  개명 및 등번호 변경 적용: {team} {old_name} → {new_name} ({renamed}행)")
        else:
            print(f"  [미발견] {team} {old_name}")
    return df

print("\n[개명 매핑 적용]")
df = apply_rename(df, RENAME_MAP)

# ============================================================
# plate_seq 재정의
# game_id + topbot 단위로 타격 결과 행에 순서대로 번호 부여
# 승부주자 포함, 순수 주루/수비 이벤트만 제외
# ============================================================
NON_BATTING_EXACT = {
    '실책',
    '홈공과어필아웃',
}

NON_BATTING_CONTAINS = [
    '도루', '도루자', '주자아웃', '오버런','런다운',
    '송구실책', '포구실책',
    '대주자', '견제사', '포일', '보크',
    '주루방해', '타구맞음', '루공과', '폭투', 
    '득점',
]

def is_batting(result):
    r = str(result).strip()
    if pd.isna(result) or r == 'nan' or r == '':
        return False
    # 쉼표 앞부분만 사용 (예: '3번R,도루' → '3번R')
    r = r.split(',')[0].strip()
    # 낫아웃은 종류 무관하게 항상 타석
    if '낫아웃' in r:
        return True
    # 괄호 안 내용 제거한 base로 판단
    # 예: '3땅R[송구실책]' → base='3땅R' → NON_BATTING_CONTAINS에 없으므로 타석
    base = re.sub(r'\[.*?\]', '', r).strip()
    if base in NON_BATTING_EXACT:
        return False
    if any(k in base for k in NON_BATTING_CONTAINS):
        return False
    return True

print("\n[plate_seq 재정의 시작]")
df['plate_seq'] = None

for i, ((game_id, topbot), grp) in enumerate(df.groupby(['game_id', 'topbot'], sort=False), 1):
    seq = 1
    for idx, row in grp.iterrows():
        if is_batting(row['result']):
            df.at[idx, 'plate_seq'] = float(seq)
            seq += 1
    if i % 500 == 0:
        print(f"  진행중: {i}게임 처리완료...")

print("plate_seq 재정의 완료")

# ============================================================
# 검증
# ============================================================
pa = df[df['plate_seq'].notna()]
print(f"\nplate_seq 부여된 행: {len(pa)}")
print(f"승부주자 포함 확인: {(pa['result'] == '승부주자').sum()}행")

# 낫아웃 종류별 확인
natout = df[df['result'].astype(str).str.contains('낫아웃', na=False)]
print(f"\n[낫아웃 종류별 plate_seq 부여 확인]")
print(natout.groupby('result')['plate_seq'].apply(
    lambda x: f"있음={x.notna().sum()} / 없음={x.isna().sum()}"
).to_string())

# 개명 후 잔존 확인
for (team, old_name), new_name in RENAME_MAP.items():
    remain = df[(df['team'] == team) & (df['batter'] == old_name)]
    print(f"\n개명 후 {old_name} 잔존 행: {len(remain)}")

# plate_seq 없는 타격 의심 행
no_seq_after = df[
    df['plate_seq'].isna() &
    df['result'].notna() &
    df['result'].apply(is_batting)
]
print(f"\n재정의 후 plate_seq 없는 타격 의심 행: {len(no_seq_after)}")
if len(no_seq_after) > 0:
    print(no_seq_after['result'].value_counts().to_string())

# ============================================================
# 저장
# ============================================================
df.to_excel(OUTPUT_FILE, index=False)
print(f"\n저장 완료: {OUTPUT_FILE}")

# 행 채우기 (주자아웃, 도루자 등 비타석의 행들을 채움)

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# ── 경로 설정
BASE_DIR    = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수"
INPUT_FILE  = f"{BASE_DIR}/cleaned_{YEAR}_merge.xlsx"
OUTPUT_FILE = f"{BASE_DIR}/cleaned_{YEAR}_merge_filled.xlsx"

# ── 데이터 로드
df = pd.read_excel(INPUT_FILE)
print(f"총 행 수: {len(df)}")
print(f"처리 전 game_id NaN 행 수: {df['game_id'].isna().sum()}")
print(f"처리 전 inning  NaN 행 수: {df['inning'].isna().sum()}")

# ── 채울 컬럼 목록
FILL_COLS = [
    'game_id', 'topbot', 'team', 'inning',
    'tournament', 'round', 'details', 'year', 'date', 'time', 'stadium',
    'team1', 'score1', 'result1', 'team2', 'score2', 'result2', 'def_team'
]

# 실제 존재하는 컬럼만 필터링 (컬럼명 오타 방지)
FILL_COLS = [c for c in FILL_COLS if c in df.columns]

# ── 대수비 단독 행 마스크
DAESUBI_ONLY = df['result'].str.strip() == '대수비'

# ── NaN 채우기: 대수비 단독 행은 임시로 제외하고 ffill, 이후 복원
# 방법: 대수비 단독 행의 game_id를 NaN으로 유지한 채 ffill → 대수비 행은 건드리지 않음
filled_count  = 0
skipped_count = 0

for i in df.index[df['game_id'].isna()]:
    if DAESUBI_ONLY.at[i]:
        skipped_count += 1
        continue
    prev_idx = i - 1
    if prev_idx < 0 or pd.isna(df.at[prev_idx, 'game_id']):
        continue
    for col in FILL_COLS:
        if col in df.columns:
            df.at[i, col] = df.at[prev_idx, col]
    filled_count += 1

print(f"\n채운 행 수   : {filled_count}")
print(f"스킵(대수비) : {skipped_count}")
print(f"처리 후 game_id NaN 행 수: {df['game_id'].isna().sum()}")
print(f"처리 후 inning  NaN 행 수: {df['inning'].isna().sum()}")

# ── 저장
df.to_excel(OUTPUT_FILE, index=False)
print(f"\n저장 완료 → {OUTPUT_FILE}")

# enter_inning_summary 파일 생성

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# ─── 경로 설정
BASE_DIR    = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/{YEAR}/검수"
INPUT_PATH  = f"{BASE_DIR}/cleaned_{YEAR}_merge_filled.xlsx"
OUTPUT_PATH = f"{BASE_DIR}/{YEAR}_enter_inning_summary.xlsx"

df = pd.read_excel(INPUT_PATH)

df['inning'] = df['inning'].astype(int)
df['outs_when_up'] = df['outs_when_up'].fillna(0).astype(int)

def get_score(row):
    if row['topbot'] == 'top':
        val = row['away_score']
    else:
        val = row['home_score']
    return val if pd.notna(val) else 0.0

df['score_this_pa'] = df.apply(get_score, axis=1)

grouped = df.groupby(['file_name', 'game_id', 'inning', 'topbot'], sort=False)
summary = grouped.agg(
    enter_out=('outs_when_up', 'sum'),
    enter_score=('score_this_pa', 'sum')
).reset_index()

summary['enter_out']   = summary['enter_out'].astype(int)
summary['enter_score'] = summary['enter_score'].astype(int)

topbot_order = {'top': 0, 'bot': 1}
summary['_tb_order'] = summary['topbot'].map(topbot_order)
summary = summary.sort_values(['game_id', '_tb_order', 'inning']).drop(columns='_tb_order')
summary = summary.reset_index(drop=True)

# ─── 엑셀 출력
wb = Workbook()
ws = wb.active
ws.title = "이닝별 요약"

header_fill   = PatternFill('solid', start_color='4472C4', end_color='4472C4')
header_font   = Font(name='Arial', bold=True, color='FFFFFF', size=10)
data_font     = Font(name='Arial', size=10)
merge_font    = Font(name='Arial', bold=True, size=10)
game_id_fill  = PatternFill('solid', start_color='DCE6F1', end_color='DCE6F1')
center_align  = Alignment(horizontal='center', vertical='center')
thin_side     = Side(style='thin', color='BFBFBF')
thin_border   = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

def merged_border(is_bot_row):
    bot_s = thin_side if is_bot_row else Side(style=None)
    return Border(left=thin_side, right=thin_side, top=Side(style=None), bottom=bot_s)

headers = ['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score']
for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_idx, value=h)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = center_align
    cell.border    = thin_border

for row_idx, row in summary.iterrows():
    excel_row  = row_idx + 2
    row_values = [row['file_name'], row['game_id'], row['inning'],
                  row['topbot'], row['enter_out'], row['enter_score']]
    for col_idx, val in enumerate(row_values, 1):
        cell = ws.cell(row=excel_row, column=col_idx, value=val)
        cell.font      = data_font
        cell.alignment = center_align
        cell.border    = thin_border

i = 0
while i < len(summary):
    current_gid  = summary.loc[i, 'game_id']
    current_file = summary.loc[i, 'file_name']
    start_excel  = i + 2
    j = i + 1
    while j < len(summary) and summary.loc[j, 'game_id'] == current_gid:
        j += 1
    end_excel = j + 1

    for col in (1, 2):
        if end_excel > start_excel:
            ws.merge_cells(start_row=start_excel, start_column=col,
                           end_row=end_excel,     end_column=col)

        top_cell           = ws.cell(row=start_excel, column=col)
        top_cell.value     = current_file if col == 1 else current_gid
        top_cell.font      = merge_font
        top_cell.fill      = game_id_fill
        top_cell.alignment = Alignment(horizontal='center', vertical='center')
        top_cell.border    = thin_border

        for r in range(start_excel + 1, end_excel + 1):
            ws.cell(row=r, column=col).border = merged_border(is_bot_row=(r == end_excel))

    i = j

col_widths = [14, 12, 8, 8, 10, 12]
for col_idx, width in enumerate(col_widths, 1):
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = width

ws.freeze_panes = 'A2'

wb.save(OUTPUT_PATH)
print(f"완료: {OUTPUT_PATH}")
print(f"  - 총 {len(summary):,}개 (game×inning×topbot) 행")
print(f"  - 총 {summary['game_id'].nunique():,}개 game_id 병합")


# real 파일 out 추가

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"

for year in range(2020, 2027):
    path = f"{BASE_DIR}/inning_summary/inning_summary_{year}.xlsx"
    try:
        df = pd.read_excel(path)
        if 'real_out' not in df.columns:
            score_idx = df.columns.get_loc('score') + 1
            df.insert(score_idx, 'real_out', 3)
        else:
            df['real_out'] = 3
        df.to_excel(path, index=False)
        print(f"✓ {year}: {len(df)}행 처리 완료")
    except FileNotFoundError:
        print(f"✗ {year}: 파일 없음 ({path})")


# enter 파일과 real 파일 대조

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# ─── 경로 설정
BASE_DIR    = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
REAL_PATH   = f"{BASE_DIR}/inning_summary/inning_summary_{YEAR}.xlsx"
ENTER_PATH  = f"{BASE_DIR}/{YEAR}/검수/{YEAR}_enter_inning_summary.xlsx"
OUTPUT_PATH = f"{BASE_DIR}/{YEAR}/검수/{YEAR}_diff_inning_summary.xlsx"

real  = pd.read_excel(REAL_PATH)
enter = pd.read_excel(ENTER_PATH)

real['game_id']    = real['game_id'].ffill().astype(int)
enter['game_id']   = enter['game_id'].ffill().astype(int)
enter['file_name'] = enter['file_name'].ffill()

merged = pd.merge(
    real[['game_id', 'inning', 'topbot', 'score', 'real_out']].rename(columns={'score': 'actual_score'}),
    enter[['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score']],
    on=['game_id', 'inning', 'topbot'],
    how='inner'
)

merged['score_diff'] = merged['enter_score'] - merged['actual_score']
merged['out_diff']   = merged['enter_out']   - merged['real_out']

merged = merged[['file_name', 'game_id', 'inning', 'topbot',
                 'actual_score', 'enter_score', 'score_diff',
                 'real_out', 'enter_out', 'out_diff']]

topbot_order = {'top': 0, 'bot': 1}
merged['_tb_order'] = merged['topbot'].map(topbot_order)
merged = merged.sort_values(['game_id', '_tb_order', 'inning']).drop(columns='_tb_order')
merged = merged.reset_index(drop=True)

# ─── 엑셀 출력
wb = Workbook()
ws = wb.active
ws.title = "이닝별 diff"

header_fill   = PatternFill('solid', start_color='4472C4', end_color='4472C4')
header_font   = Font(name='Arial', bold=True, color='FFFFFF', size=10)
data_font     = Font(name='Arial', size=10)
merge_font    = Font(name='Arial', bold=True, size=10)
merge_fill    = PatternFill('solid', start_color='DCE6F1', end_color='DCE6F1')
zero_fill     = PatternFill('solid', start_color='C6EFCE', end_color='C6EFCE')
nonzero_fill  = PatternFill('solid', start_color='FFD7D7', end_color='FFD7D7')
center_align  = Alignment(horizontal='center', vertical='center')
thin_side     = Side(style='thin', color='BFBFBF')
thin_border   = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

def merged_border(is_bot_row):
    bot_s = thin_side if is_bot_row else Side(style=None)
    return Border(left=thin_side, right=thin_side, top=Side(style=None), bottom=bot_s)

headers = ['file_name', 'game_id', 'inning', 'topbot',
           'actual_score', 'enter_score', 'score_diff',
           'real_out', 'enter_out', 'out_diff']
for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_idx, value=h)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = center_align
    cell.border    = thin_border

for row_idx, row in merged.iterrows():
    excel_row  = row_idx + 2
    row_values = [row['file_name'], row['game_id'], row['inning'], row['topbot'],
                  row['actual_score'], row['enter_score'], row['score_diff'],
                  row['real_out'], row['enter_out'], row['out_diff']]
    for col_idx, val in enumerate(row_values, 1):
        cell = ws.cell(row=excel_row, column=col_idx, value=val)
        cell.font      = data_font
        cell.alignment = center_align
        cell.border    = thin_border

    # score_diff(7열), out_diff(10열) 색상
    ws.cell(row=excel_row, column=7).fill  = zero_fill if row['score_diff'] == 0 else nonzero_fill
    ws.cell(row=excel_row, column=10).fill = zero_fill if row['out_diff']   == 0 else nonzero_fill

i = 0
while i < len(merged):
    current_gid  = merged.loc[i, 'game_id']
    current_file = merged.loc[i, 'file_name']
    start_excel  = i + 2
    j = i + 1
    while j < len(merged) and merged.loc[j, 'game_id'] == current_gid:
        j += 1
    end_excel = j + 1

    for col, val in [(1, current_file), (2, current_gid)]:
        if end_excel > start_excel:
            ws.merge_cells(start_row=start_excel, start_column=col,
                           end_row=end_excel,     end_column=col)

        top_cell           = ws.cell(row=start_excel, column=col)
        top_cell.value     = val
        top_cell.font      = merge_font
        top_cell.fill      = merge_fill
        top_cell.alignment = Alignment(horizontal='center', vertical='center')
        top_cell.border    = thin_border

        for r in range(start_excel + 1, end_excel + 1):
            ws.cell(row=r, column=col).border = merged_border(is_bot_row=(r == end_excel))

    i = j

col_widths = [14, 12, 8, 8, 14, 12, 11, 10, 10, 10]
for col_idx, width in enumerate(col_widths, 1):
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = width

ws.freeze_panes = 'A2'

wb.save(OUTPUT_PATH)
print(f"완료: {OUTPUT_PATH}")
print(f"  - 총 {len(merged):,}개 행")
print(f"  - 매칭된 game_id 수: {merged['game_id'].nunique():,}개")

score_diff_games = merged[merged['score_diff'] != 0]['game_id'].unique()
out_diff_games   = merged[merged['out_diff']   != 0]['game_id'].unique()
print(f"\n  - score_diff != 0 행 수: {(merged['score_diff'] != 0).sum():,}개")
print(f"    game_id 목록: {sorted(score_diff_games.tolist())}")
print(f"\n  - out_diff != 0 행 수: {(merged['out_diff'] != 0).sum():,}개")
print(f"    game_id 목록: {sorted(out_diff_games.tolist())}")


In [ ]:
print(real.head())

# 파일별 통합 파이프라인 (콘솔 로그로 검수)

In [ ]:
import pandas as pd
import os
import re
import unicodedata
import time
from google.colab import drive
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# ── 경로 설정
YEAR     = 2026
BASE_DIR = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
YEAR_DIR = f"{BASE_DIR}/{YEAR}"
REAL_PATH   = f"{BASE_DIR}/inning_summary/inning_summary_{YEAR}.xlsx"
SUMMARY_OUT = f"{YEAR_DIR}/검수/{YEAR}_enter_inning_summary.xlsx"
DIFF_OUT    = f"{YEAR_DIR}/검수/{YEAR}_diff_inning_summary.xlsx"
ISSUES_OUT  = f"{YEAR_DIR}/검수/{YEAR}_issues.xlsx"  # 오타/이상 항목 목록

os.makedirs(f"{YEAR_DIR}/검수", exist_ok=True)

# ── 공통 스타일
_header_fill  = PatternFill('solid', start_color='4472C4', end_color='4472C4')
_issues_fill  = PatternFill('solid', start_color='C55A11', end_color='C55A11')  # 주황 (issues)
_header_font  = Font(name='Arial', bold=True, color='FFFFFF', size=10)
_data_font    = Font(name='Arial', size=10)
_merge_font   = Font(name='Arial', bold=True, size=10)
_merge_fill   = PatternFill('solid', start_color='DCE6F1', end_color='DCE6F1')
_green_fill   = PatternFill('solid', start_color='C6EFCE', end_color='C6EFCE')
_red_fill     = PatternFill('solid', start_color='FFD7D7', end_color='FFD7D7')
_center       = Alignment(horizontal='center', vertical='center')
_left         = Alignment(horizontal='left',   vertical='center')
_thin         = Side(style='thin', color='BFBFBF')
_border       = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)

def _merged_border(is_last_row):
    bot = _thin if is_last_row else Side(style=None)
    return Border(left=_thin, right=_thin, top=Side(style=None), bottom=bot)

def _write_header(ws, headers, fill):
    for c, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=c, value=h)
        cell.font = _header_font; cell.fill = fill
        cell.alignment = _center; cell.border = _border

def _merge_col(ws, col, val, start_row, end_row):
    if end_row > start_row:
        ws.merge_cells(start_row=start_row, start_column=col,
                       end_row=end_row,     end_column=col)
    cell = ws.cell(row=start_row, column=col, value=val)
    cell.font = _merge_font; cell.fill = _merge_fill
    cell.alignment = _center; cell.border = _border
    for r in range(start_row + 1, end_row + 1):
        ws.cell(row=r, column=col).border = _merged_border(r == end_row)

# issues DataFrame 컬럼 정의 (모든 step에서 동일하게 사용)
_ISSUE_COLS = ['game_id', '단계', '문제', '상세']

# ════════════════════════════════════════════════════════════════
# Step 1: 쉼표 삭제
#   이슈 반환: 낫아웃인데 쉼표가 남은 행 (game_id, result)
# ════════════════════════════════════════════════════════════════
_remove_always   = ['대수비', '대타']
_prefixes        = ['포구실책', '송구실책', '실책', '포일', '폭투', '보크']
_NATOUT_SUFFIXES = ['폭투', '포일', '송구실책', '포구실책', '실책']

def _is_prefix_only(p):
    base = re.sub(r'\[.*?\]', '', p).strip()
    return any(pf in base for pf in _prefixes)

def _clean_result(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    value = value.replace('.', ',').replace(';', ',')
    if ',' not in value:
        return float('nan') if any(r in value for r in _remove_always) else value
    parts = [p.strip() for p in value.split(',')]
    if all(any(r in p for r in _remove_always) for p in parts):
        return parts[0]
    parts = [p for p in parts if not any(r in p for r in _remove_always)] or parts
    natout = next((p for p in parts if '낫아웃' in p), None)
    if natout:
        if natout == '낫아웃+':
            suf = next((p for p in parts if p != natout and any(s in p for s in _NATOUT_SUFFIXES)), None)
            if suf:
                return next((f'낫아웃+{s}' for s in _NATOUT_SUFFIXES if s in suf), '낫아웃+')
            return '낫아웃+'
        if any(s in natout for s in _NATOUT_SUFFIXES) or '낫아웃-' in natout:
            return natout
        return natout
    non_pref = [p for p in parts if not _is_prefix_only(p)]
    return non_pref[0] if non_pref else parts[0]

def step1_clean(df):
    solo_mask = df['result'].str.strip() == '대수비'
    n_del = solo_mask.sum()
    df = df[~solo_mask].copy().reset_index(drop=True)

    before = df['result'].copy()
    df['result'] = df['result'].apply(_clean_result)
    n_changed = (df['result'].astype(str) != before.astype(str)).sum()

    natout_bad = df[
        df['result'].astype(str).str.contains('낫아웃', na=False) &
        df['result'].astype(str).str.contains(',', na=False)
    ]
    print(f"  단독 대수비 삭제: {n_del}행 | 쉼표 처리: {n_changed}건", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(natout_bad):
        print(f"\n  ⚠ [낫아웃 미처리 의심] {len(natout_bad)}건 → {natout_bad['result'].tolist()}", end='')
        tmp = natout_bad[['game_id', 'result']].copy().rename(columns={'result': '상세'})
        tmp['단계'] = '[1] 쉼표 삭제'
        tmp['문제'] = '낫아웃 미처리 의심'
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 2: 재정의 (plate_seq)
#   이슈 반환: 타석 의심인데 plate_seq 없는 행 (game_id, result)
# ════════════════════════════════════════════════════════════════
_NON_BATTING_EXACT    = {'실책', '홈공과어필아웃'}
_NON_BATTING_CONTAINS = [
    '도루', '도루자', '주자아웃', '오버런', '런다운',
    '송구실책', '포구실책', '대주자', '견제사', '포일',
    '보크', '주루방해', '타구맞음', '루공과', '폭투', '득점',
]

def _is_batting(result):
    r = str(result).strip()
    if pd.isna(result) or r in ('nan', ''):
        return False
    r = r.split(',')[0].strip()
    if '낫아웃' in r:
        return True
    base = re.sub(r'\[.*?\]', '', r).strip()
    return base not in _NON_BATTING_EXACT and not any(k in base for k in _NON_BATTING_CONTAINS)

def step2_redefine(df):
    RENAME_MAP = {}
    for (team, old), new in RENAME_MAP.items():
        mask = (df['team'] == team) & (df['batter'] == old)
        if mask.any():
            df.loc[mask, 'batter'] = new

    df['plate_seq'] = None
    for (game_id, topbot), grp in df.groupby(['game_id', 'topbot'], sort=False):
        seq = 1
        for idx, row in grp.iterrows():
            if _is_batting(row['result']):
                df.at[idx, 'plate_seq'] = float(seq)
                seq += 1

    n_seq  = df['plate_seq'].notna().sum()
    no_seq = df[df['plate_seq'].isna() & df['result'].notna() & df['result'].apply(_is_batting)]

    print(f"  plate_seq 부여: {n_seq}행", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(no_seq):
        gids = no_seq['game_id'].unique().tolist()
        vals = no_seq['result'].value_counts().head(3).to_dict()
        print(f"\n  ⚠ [plate_seq 누락 의심] {len(no_seq)}건 → game_id: {gids}\n    result 샘플: {vals}", end='')
        tmp = no_seq[['game_id', 'result']].copy().rename(columns={'result': '상세'})
        tmp['단계'] = '[2] 재정의'
        tmp['문제'] = 'plate_seq 누락 의심'
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 3: 행 채우기
#   이슈 반환: 채운 뒤에도 game_id/inning NaN 잔존 행
# ════════════════════════════════════════════════════════════════
_FILL_COLS = [
    'game_id', 'topbot', 'team', 'inning',
    'tournament', 'round', 'details', 'year', 'date', 'time', 'stadium',
    'team1', 'score1', 'result1', 'team2', 'score2', 'result2', 'def_team'
]

def step3_fill(df):
    fill_cols    = [c for c in _FILL_COLS if c in df.columns]
    daesubi_only = df['result'].str.strip() == '대수비'
    filled = skipped = 0

    for i in df.index[df['game_id'].isna()]:
        if daesubi_only.at[i]:
            skipped += 1
            continue
        prev = i - 1
        if prev < 0 or pd.isna(df.at[prev, 'game_id']):
            continue
        for col in fill_cols:
            df.at[i, col] = df.at[prev, col]
        filled += 1

    nan_game_id = int(df['game_id'].isna().sum())
    nan_inning  = int(df['inning'].isna().sum())

    print(f"  채운 행: {filled} | 스킵(대수비): {skipped}", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if nan_game_id or nan_inning:
        print(f"\n  ⚠ [game_id NaN 잔존] {nan_game_id}행 / [inning NaN 잔존] {nan_inning}행", end='')
        nan_rows = df[df['game_id'].isna()][['result']].copy()
        nan_rows['game_id'] = pd.NA
        nan_rows['단계'] = '[3] 행 채우기'
        nan_rows['문제'] = 'NaN 잔존 (game_id 미확인)'
        nan_rows['상세'] = nan_rows['result'].astype(str)
        issues = nan_rows[_ISSUE_COLS].reset_index(drop=True)
    print()
    return df, issues

# ════════════════════════════════════════════════════════════════
# Step 4: enter_inning_summary 계산
#   이슈 반환: enter_out=0인 이닝 (game_id, inning, topbot)
# ════════════════════════════════════════════════════════════════
def step4_summary(df):
    df['inning']       = df['inning'].fillna(0).astype(int)
    df['outs_when_up'] = df['outs_when_up'].fillna(0).astype(int)

    def _get_score(row):
        col = 'away_score' if row['topbot'] == 'top' else 'home_score'
        v   = row.get(col, None)
        return v if pd.notna(v) else 0.0

    df['score_this_pa'] = df.apply(_get_score, axis=1)

    summary = df.groupby(['file_name', 'game_id', 'inning', 'topbot'], sort=False).agg(
        enter_out  =('outs_when_up',  'sum'),
        enter_score=('score_this_pa', 'sum')
    ).reset_index()
    summary['enter_out']   = summary['enter_out'].astype(int)
    summary['enter_score'] = summary['enter_score'].astype(int)

    n_games  = summary['game_id'].nunique()
    n_inn    = len(summary)
    zero_out = summary[summary['enter_out'] == 0]

    print(f"  {n_games}경기 / {n_inn}이닝 처리", end='')

    issues = pd.DataFrame(columns=_ISSUE_COLS)
    if len(zero_out):
        gids = zero_out['game_id'].unique().tolist()
        print(f"\n  ⚠ [enter_out=0 이닝] {len(zero_out)}건 → game_id: {gids}", end='')
        tmp = zero_out[['game_id', 'inning', 'topbot']].copy()
        tmp['단계'] = '[4] enter_summary'
        tmp['문제'] = 'enter_out=0'
        tmp['상세'] = tmp['inning'].astype(str) + '이닝 ' + tmp['topbot']
        issues = tmp[_ISSUE_COLS].reset_index(drop=True)
    print()
    return summary, issues

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: enter_inning_summary
# ════════════════════════════════════════════════════════════════
def save_summary_xlsx(df, path):
    headers = ['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score']
    wb = Workbook(); ws = wb.active; ws.title = "이닝별 요약"
    _write_header(ws, headers, _header_fill)
    for i, row in df.iterrows():
        er = i + 2
        for c, h in enumerate(headers, 1):
            cell = ws.cell(row=er, column=c, value=row[h])
            cell.font = _data_font; cell.alignment = _center; cell.border = _border
    i = 0
    while i < len(df):
        gid = df.iloc[i]['game_id']; fname = df.iloc[i]['file_name']
        j = i + 1
        while j < len(df) and df.iloc[j]['game_id'] == gid: j += 1
        _merge_col(ws, 1, fname, i + 2, j + 1)
        _merge_col(ws, 2, gid,   i + 2, j + 1)
        i = j
    for ci, w in enumerate([14, 12, 8, 8, 10, 12], 1):
        ws.column_dimensions[ws.cell(1, ci).column_letter].width = w
    ws.freeze_panes = 'A2'
    wb.save(path)

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: diff_inning_summary
# ════════════════════════════════════════════════════════════════
def save_diff_xlsx(df, path):
    headers = ['file_name', 'game_id', 'inning', 'topbot',
               'actual_score', 'enter_score', 'score_diff',
               'real_out', 'enter_out', 'out_diff']
    wb = Workbook(); ws = wb.active; ws.title = "이닝별 diff"
    _write_header(ws, headers, _header_fill)
    for i, row in df.iterrows():
        er = i + 2
        for c, h in enumerate(headers, 1):
            cell = ws.cell(row=er, column=c, value=row[h])
            cell.font = _data_font; cell.alignment = _center; cell.border = _border
        ws.cell(er, 7).fill  = _green_fill if row['score_diff'] == 0 else _red_fill
        ws.cell(er, 10).fill = _green_fill if row['out_diff']   == 0 else _red_fill
    i = 0
    while i < len(df):
        gid = df.iloc[i]['game_id']; fname = df.iloc[i]['file_name']
        j = i + 1
        while j < len(df) and df.iloc[j]['game_id'] == gid: j += 1
        _merge_col(ws, 1, fname, i + 2, j + 1)
        _merge_col(ws, 2, gid,   i + 2, j + 1)
        i = j
    for ci, w in enumerate([14, 12, 8, 8, 14, 12, 11, 10, 10, 10], 1):
        ws.column_dimensions[ws.cell(1, ci).column_letter].width = w
    ws.freeze_panes = 'A2'
    wb.save(path)

# ════════════════════════════════════════════════════════════════
# 스타일 엑셀 저장: issues (오타/이상 항목 목록)
#   헤더: 주황색 / 열: file_name, game_id, 단계, 문제, 상세
# ════════════════════════════════════════════════════════════════
def save_issues_xlsx(df, path):
    headers = ['file_name', 'game_id', '단계', '문제', '상세']
    wb = Workbook(); ws = wb.active; ws.title = "이슈 목록"
    _write_header(ws, headers, _issues_fill)
    for i, row in df.iterrows():
        er = i + 2
        for c, h in enumerate(headers, 1):
            val  = row.get(h, '')
            cell = ws.cell(row=er, column=c, value=('' if pd.isna(val) else val))
            cell.font      = _data_font
            cell.alignment = _left if h == '상세' else _center
            cell.border    = _border
    for ci, w in enumerate([16, 10, 16, 22, 40], 1):
        ws.column_dimensions[ws.cell(1, ci).column_letter].width = w
    ws.freeze_panes = 'A2'
    wb.save(path)

print("✓ 함수 정의 완료")

In [ ]:
# ════════════════════════════════════════════
# 메인 파이프라인 실행
# ════════════════════════════════════════════
def run_pipeline():

    # ── 소스 파일 수집: YEAR_DIR 바로 아래 N월 폴더만 탐색
    source_files = []
    month_pat = re.compile(r'^\d+월$')
    file_pat  = re.compile(rf'^{YEAR}_\d{{2}}_[a-z]+\.xlsx$')

    for entry in sorted(os.listdir(YEAR_DIR)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(YEAR_DIR, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if file_pat.match(f_nfc):
                source_files.append(os.path.join(month_dir, f_nfc))
    source_files.sort()
    print(f"총 {len(source_files)}개 파일 발견\n")

    all_summaries = []
    all_issues    = []   # 각 step에서 수집된 이슈 DataFrame 목록
    problem_files = []
    t_total = time.time()

    # ── 파일별 처리 루프
    for idx, fpath in enumerate(source_files, 1):
        fname    = unicodedata.normalize('NFC', os.path.basename(fpath))
        stem     = os.path.splitext(fname)[0]
        problems = []

        print("=" * 60)
        print(f"[{idx}/{len(source_files)}] {fname}")
        print("-" * 60)
        t0 = time.time()

        try:
            df = pd.read_excel(fpath)
            df['file_name'] = stem
            print(f"  로드: {len(df)}행")
        except Exception as e:
            print(f"  ❌ 로드 실패: {e}")
            problem_files.append((fname, [f"로드 실패: {e}"]))
            continue

        # Step 1: 쉼표 삭제
        print("[1] 쉼표 삭제")
        df, iss1 = step1_clean(df)
        if len(iss1):
            iss1.insert(0, 'file_name', stem)
            all_issues.append(iss1)
            problems.append(f"낫아웃 미처리 {len(iss1)}건")

        # Step 2: plate_seq 재정의
        print("[2] 재정의 (plate_seq)")
        df, iss2 = step2_redefine(df)
        if len(iss2):
            iss2.insert(0, 'file_name', stem)
            all_issues.append(iss2)
            problems.append(f"plate_seq 누락 {len(iss2)}건")

        # Step 3: 비타석 행 채우기
        print("[3] 행 채우기")
        df, iss3 = step3_fill(df)
        if len(iss3):
            iss3.insert(0, 'file_name', stem)
            all_issues.append(iss3)
            problems.append(f"NaN 잔존 {len(iss3)}행")

        # Step 4: 이닝별 요약 계산
        print("[4] enter_inning_summary")
        summary, iss4 = step4_summary(df)
        all_summaries.append(summary)
        if len(iss4):
            iss4.insert(0, 'file_name', stem)
            all_issues.append(iss4)
            problems.append(f"enter_out=0 이닝 {len(iss4)}건")

        elapsed = time.time() - t0
        status  = "✅" if not problems else "⚠"
        extra   = f" | 문제: {', '.join(problems)}" if problems else ""
        print(f"{status} {fname} 완료 ({elapsed:.1f}초){extra}")
        if problems:
            problem_files.append((fname, problems))

    # ── 전체 합본 → 스타일 엑셀 저장
    print("\n" + "=" * 60)
    print("enter_inning_summary 저장 중...")
    full = pd.concat(all_summaries, ignore_index=True)
    full['_tb'] = full['topbot'].map({'top': 0, 'bot': 1})
    full = full.sort_values(['game_id', '_tb', 'inning']).drop(columns='_tb').reset_index(drop=True)
    save_summary_xlsx(full, SUMMARY_OUT)
    print(f"  → {SUMMARY_OUT}")

    # ── real 파일과 대조
    print("real 파일 대조 중...")
    real = pd.read_excel(REAL_PATH)
    real['game_id'] = real['game_id'].ffill().astype(int)

    merged = pd.merge(
        real[['game_id', 'inning', 'topbot', 'score', 'real_out']].rename(columns={'score': 'actual_score'}),
        full[['file_name', 'game_id', 'inning', 'topbot', 'enter_out', 'enter_score']],
        on=['game_id', 'inning', 'topbot'],
        how='inner'
    )
    merged['score_diff'] = merged['enter_score'] - merged['actual_score']
    merged['out_diff']   = merged['enter_out']   - merged['real_out']
    merged = merged.reset_index(drop=True)
    save_diff_xlsx(merged, DIFF_OUT)
    print(f"  → {DIFF_OUT}")

    # ── 대조 결과도 issues에 추가
    # diff != 0 = 입력값과 실제값 불일치 → 해당 파일·경기 수정 필요
    score_bad = merged[merged['score_diff'] != 0]
    out_bad   = merged[merged['out_diff']   != 0]

    if len(score_bad):
        si = score_bad[['file_name', 'game_id', 'inning', 'topbot', 'score_diff']].copy()
        si['단계'] = '[대조] score'
        si['문제'] = 'score_diff != 0'
        si['상세'] = si['inning'].astype(str) + '이닝 ' + si['topbot'] + ' (diff=' + si['score_diff'].astype(str) + ')'
        all_issues.append(si[['file_name', 'game_id', '단계', '문제', '상세']])

    if len(out_bad):
        oi = out_bad[['file_name', 'game_id', 'inning', 'topbot', 'out_diff']].copy()
        oi['단계'] = '[대조] out'
        oi['문제'] = 'out_diff != 0'
        oi['상세'] = oi['inning'].astype(str) + '이닝 ' + oi['topbot'] + ' (diff=' + oi['out_diff'].astype(str) + ')'
        all_issues.append(oi[['file_name', 'game_id', '단계', '문제', '상세']])

    # ── issues 파일 저장
    # file_name 열로 원본 파일(예: 2026/4월/2026_04_a.xlsx) 바로 찾아갈 수 있음
    print("issues 파일 저장 중...")
    if all_issues:
        issues_df = pd.concat(all_issues, ignore_index=True)
        issues_df = issues_df[['file_name', 'game_id', '단계', '문제', '상세']]
        save_issues_xlsx(issues_df, ISSUES_OUT)
        print(f"  → {ISSUES_OUT} ({len(issues_df)}건)")
    else:
        print("  ✅ 이슈 없음 — issues 파일 미생성")

    # ── 최종 요약 출력
    total_t = time.time() - t_total

    print("\n" + "=" * 60)
    print("전체 요약")
    print("-" * 60)
    print(f"처리 파일: {len(source_files)}개 | 소요: {total_t:.1f}초")

    if problem_files:
        print(f"\n⚠ 문제 발생 파일: {len(problem_files)}개")
        for fname, probs in problem_files:
            print(f"  - {fname}: {', '.join(probs)}")
    else:
        print("\n✅ 모든 파일 정상 처리")

    print(f"\n[최종 대조] 매칭 경기: {merged['game_id'].nunique():,}개")
    print(f"  score_diff != 0: {len(score_bad)}건 → {sorted(score_bad['game_id'].unique().tolist())}")
    print(f"  out_diff   != 0: {len(out_bad)}건 → {sorted(out_bad['game_id'].unique().tolist())}")
    print("=" * 60)

run_pipeline()